In [15]:
import pandas as pd
from surprise import SVD, Dataset, Reader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

# === Load dữ liệu đánh giá ===
ratings = pd.read_csv("ratings.csv")[['UserId', 'MediaId', 'RatingValue']]
reader = Reader(rating_scale=(1, 10))
data = Dataset.load_from_df(ratings, reader)

# === Train mô hình Collaborative Filtering (SVD) ===
trainset = data.build_full_trainset()
svd_model = SVD()
svd_model.fit(trainset)

print("✅ CF model trained!")

# === Load dữ liệu phim (cho Content-Based) ===
movies = pd.read_csv("movies_tv_combined.csv")

# Gộp thông tin text để tính TF-IDF
movies['combined_features'] = (
    movies['Title'].fillna('') + ' ' +
    movies['Overview'].fillna('') + ' ' +
    movies['Genres'].fillna('') + ' ' +
    movies['Director'].fillna('') + ' ' +
    movies['Studio'].fillna('')
)

vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(movies['combined_features'])

# Tính độ tương đồng cosine giữa các phim
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print("✅ Content-based features processed!")

# === Lưu model, vectorizer, và cosine matrix ===
with open("recommend_model.pkl", "wb") as f:
    pickle.dump({
        "svd_model": svd_model,
        "vectorizer": vectorizer,
        "cosine_sim": cosine_sim,
        "movies": movies
    }, f)

print("💾 Saved all to recommend_model.pkl")


✅ CF model trained!
✅ Content-based features processed!
💾 Saved all to recommend_model.pkl
